# Text-to-SQL Agent: Natural Language to SQL with Safety & Validation

**Goal:** build an agent that translates natural-language questions into SQL queries, validates and executes them safely, and summarizes the results in plain English.

---

## Why Text-to-SQL?

SQL is the lingua franca of structured data. Most business data lives in databases. But writing SQL requires technical knowledge most users don't have. A text-to-SQL agent bridges that gap — the user asks a question, the agent figures out the SQL, and returns a plain-English answer.

## The Challenge: Safety

Text-to-SQL is powerful but dangerous. Without guardrails:
- A user could ask something that triggers `DROP TABLE`
- A poorly generated query could return millions of rows
- A confused agent could expose data the user shouldn't see

**SQL safety is not optional.** This notebook covers it in depth.

## Architecture

```text
User question
      │
      ▼
┌─────────────┐     ┌─────────────────┐
│  NLU Parser  │────►│  SQL Generator  │  Build a candidate SQL query
└─────────────┘     └────────┬────────┘
                             │
                             ▼
                    ┌─────────────────┐
                    │  SQL Validator  │  Safety + syntax + schema checks
                    └────────┬────────┘
                             │ PASS / FAIL
                             ▼
                    ┌─────────────────┐
                    │  SQL Executor   │  Run against SQLite DB
                    └────────┬────────┘
                             │ ResultSet
                             ▼
                    ┌─────────────────┐
                    │  Summarizer     │  Plain-English answer + table
                    └─────────────────┘
```


## 1. Environment Setup

In [ ]:
!pip install -q pandas numpy matplotlib seaborn sqlparse

In [ ]:
import json
import random
import re
import sqlite3
import textwrap
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    import sqlparse
    HAS_SQLPARSE = True
except ImportError:
    HAS_SQLPARSE = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = Path.cwd()
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
DB_PATH = ARTIFACT_DIR / "sales.db"

print(f"Python SQLite version: {sqlite3.sqlite_version}")
print(f"sqlparse available:    {HAS_SQLPARSE}")
print(f"DB path:               {DB_PATH}")

## 2. The Database

We create a realistic SQLite database with four related tables representing a retail business. This gives us a rich schema for demonstrating joins, aggregations, filters, and date arithmetic.

```
customers ──┐
             ├── orders ──┐
products ───┘              ├── order_items
                           │
             sales_reps ───┘
```


In [ ]:
def build_database(db_path: Path) -> None:
    """Create and populate the sales database."""
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    # ── Schema ──
    cur.executescript("""
    DROP TABLE IF EXISTS order_items;
    DROP TABLE IF EXISTS orders;
    DROP TABLE IF EXISTS customers;
    DROP TABLE IF EXISTS products;
    DROP TABLE IF EXISTS sales_reps;

    CREATE TABLE customers (
        customer_id   INTEGER PRIMARY KEY,
        name          TEXT NOT NULL,
        email         TEXT UNIQUE,
        region        TEXT NOT NULL,
        tier          TEXT NOT NULL,   -- 'Bronze', 'Silver', 'Gold', 'Platinum'
        signup_date   DATE NOT NULL
    );

    CREATE TABLE products (
        product_id    INTEGER PRIMARY KEY,
        name          TEXT NOT NULL,
        category      TEXT NOT NULL,
        unit_price    REAL NOT NULL,
        cost          REAL NOT NULL
    );

    CREATE TABLE sales_reps (
        rep_id        INTEGER PRIMARY KEY,
        name          TEXT NOT NULL,
        region        TEXT NOT NULL,
        quota         REAL NOT NULL
    );

    CREATE TABLE orders (
        order_id      INTEGER PRIMARY KEY,
        customer_id   INTEGER NOT NULL REFERENCES customers(customer_id),
        rep_id        INTEGER REFERENCES sales_reps(rep_id),
        order_date    DATE NOT NULL,
        status        TEXT NOT NULL,   -- 'completed', 'pending', 'cancelled'
        discount_pct  REAL DEFAULT 0
    );

    CREATE TABLE order_items (
        item_id       INTEGER PRIMARY KEY,
        order_id      INTEGER NOT NULL REFERENCES orders(order_id),
        product_id    INTEGER NOT NULL REFERENCES products(product_id),
        quantity      INTEGER NOT NULL,
        unit_price    REAL NOT NULL
    );
    """)

    # ── Seed data ──
    rng = np.random.default_rng(SEED)

    regions = ["North", "South", "East", "West"]
    tiers   = ["Bronze", "Silver", "Gold", "Platinum"]
    statuses = ["completed", "pending", "cancelled"]

    # Customers
    customers = [(
        i, f"Customer {i}", f"c{i}@example.com",
        rng.choice(regions), rng.choice(tiers, p=[0.4, 0.3, 0.2, 0.1]),
        (datetime(2021, 1, 1) + timedelta(days=int(rng.integers(0, 1460)))).strftime("%Y-%m-%d"),
    ) for i in range(1, 201)]
    cur.executemany("INSERT INTO customers VALUES (?,?,?,?,?,?)", customers)

    # Products
    products_data = [
        (1, "Widget A",    "Widgets",  49.99,  18.0),
        (2, "Widget B",    "Widgets",  89.99,  32.0),
        (3, "Gadget X",    "Gadgets",  149.99, 55.0),
        (4, "Gadget Y",    "Gadgets",  249.99, 90.0),
        (5, "Service Z",   "Services", 399.99, 25.0),
        (6, "Part Alpha",  "Parts",    19.99,  6.0),
        (7, "Part Beta",   "Parts",    34.99,  12.0),
        (8, "Premium Kit", "Kits",     599.99, 180.0),
    ]
    cur.executemany("INSERT INTO products VALUES (?,?,?,?,?)", products_data)

    # Sales reps
    sales_reps_data = [(i, f"Rep {i}", rng.choice(regions), float(rng.integers(50000, 200000)))
                       for i in range(1, 11)]
    cur.executemany("INSERT INTO sales_reps VALUES (?,?,?,?)", sales_reps_data)

    # Orders
    orders = []
    for i in range(1, 1001):
        cust_id = int(rng.integers(1, 201))
        rep_id  = int(rng.integers(1, 11))
        date    = (datetime(2023, 1, 1) + timedelta(days=int(rng.integers(0, 730)))).strftime("%Y-%m-%d")
        status  = rng.choice(statuses, p=[0.75, 0.15, 0.10])
        disc    = float(rng.choice([0, 5, 10, 15, 20], p=[0.5, 0.2, 0.15, 0.1, 0.05]))
        orders.append((i, cust_id, rep_id, date, status, disc))
    cur.executemany("INSERT INTO orders VALUES (?,?,?,?,?,?)", orders)

    # Order items
    items = []
    item_id = 1
    for order_id, _, _, _, _, disc in orders:
        n_items = int(rng.integers(1, 5))
        for _ in range(n_items):
            prod_id  = int(rng.integers(1, 9))
            qty      = int(rng.integers(1, 20))
            price    = products_data[prod_id - 1][3] * (1 - disc / 100)
            items.append((item_id, order_id, prod_id, qty, round(price, 2)))
            item_id += 1
    cur.executemany("INSERT INTO order_items VALUES (?,?,?,?,?)", items)

    conn.commit()
    conn.close()


build_database(DB_PATH)

# Quick sanity check
conn = sqlite3.connect(DB_PATH)
for table in ["customers", "products", "sales_reps", "orders", "order_items"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table:15s}: {n:,} rows")
conn.close()

## 3. Schema Introspection

The agent needs to know the database schema to write correct SQL. It reads the schema automatically — this is called **schema introspection** or **schema-linking**.

Schema information is injected into the prompt so the LLM (or our rule-based generator) knows what tables, columns, and relationships exist.


In [ ]:
def get_schema(db_path: Path) -> dict:
    """Introspect database schema: tables, columns, types, foreign keys."""
    conn = sqlite3.connect(db_path)
    schema = {}

    tables = [row[0] for row in
              conn.execute("SELECT name FROM sqlite_master WHERE type='table'")]

    for table in tables:
        # Columns
        cols = conn.execute(f"PRAGMA table_info({table})").fetchall()
        # Foreign keys
        fks = conn.execute(f"PRAGMA foreign_key_list({table})").fetchall()

        schema[table] = {
            "columns": [
                {"name": c[1], "type": c[2], "not_null": bool(c[3]), "pk": bool(c[5])}
                for c in cols
            ],
            "foreign_keys": [
                {"from_col": fk[3], "to_table": fk[2], "to_col": fk[4]}
                for fk in fks
            ],
            "row_count": conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0],
        }

    conn.close()
    return schema


DB_SCHEMA = get_schema(DB_PATH)

# Pretty print the schema
for table, info in DB_SCHEMA.items():
    cols_str = ", ".join(
        f"{c['name']} ({c['type']}{'*' if c['pk'] else ''})"
        for c in info["columns"]
    )
    fk_str = ", ".join(
        f"{fk['from_col']}→{fk['to_table']}.{fk['to_col']}"
        for fk in info["foreign_keys"]
    )
    print(f"{table:15s} [{info['row_count']} rows]")
    print(f"  Columns: {cols_str}")
    if fk_str:
        print(f"  FK:      {fk_str}")
    print()

In [ ]:
def schema_to_prompt(schema: dict) -> str:
    """Convert schema dict to a compact string for LLM prompts."""
    lines = ["Database schema:\n"]
    for table, info in schema.items():
        col_parts = []
        for c in info["columns"]:
            marker = " PK" if c["pk"] else ""
            col_parts.append(f"  {c['name']} {c['type']}{marker}")
        fk_lines = [f"  -- FK: {fk['from_col']} → {fk['to_table']}.{fk['to_col']}"
                    for fk in info["foreign_keys"]]
        lines.append(f"TABLE {table} (")
        lines.extend(col_parts)
        lines.extend(fk_lines)
        lines.append(")")
        lines.append("")
    return "\n".join(lines)


print(schema_to_prompt(DB_SCHEMA))

## 4. SQL Safety — The Core Concern

### What Can Go Wrong?

SQL injection and unsafe query generation are among the most serious risks in any text-to-SQL system.

#### Attack Categories

| Category | Example | Risk |
|---|---|---|
| **DDL injection** | `DROP TABLE customers` | Destroy data |
| **DML injection** | `DELETE FROM orders WHERE 1=1` | Delete all rows |
| **Data exfiltration** | `SELECT * FROM customers LIMIT 1000000` | Dump entire DB |
| **Privilege escalation** | `ATTACH DATABASE '/etc/passwd' AS secret` | Access system files |
| **Multi-statement** | `SELECT 1; DROP TABLE orders` | Execute arbitrary statements |
| **Comment injection** | `SELECT * -- ignore the rest` | Bypass intent |
| **UNION injection** | `SELECT name FROM products UNION SELECT password FROM users` | Cross-table data theft |

### Defence Layers

```text
Layer 1: Allow-list of permitted query types  (SELECT only)
Layer 2: Block dangerous keywords             (DROP, DELETE, UPDATE, ...)
Layer 3: Enforce row limits                   (LIMIT clause required)
Layer 4: Schema validation                    (only known tables/columns)
Layer 5: Parameterised execution              (no string interpolation)
Layer 6: Read-only connection                 (DB opened in immutable mode)
```

All six layers must be active. One layer is never enough.


In [ ]:
# ── Layer definitions ──

# Layer 1: Only these statement types are allowed
ALLOWED_STATEMENT_TYPES = {"SELECT"}

# Layer 2: Forbidden keywords (case-insensitive)
FORBIDDEN_KEYWORDS = {
    # DDL
    "DROP", "CREATE", "ALTER", "TRUNCATE", "RENAME",
    # DML
    "INSERT", "UPDATE", "DELETE", "MERGE", "REPLACE", "UPSERT",
    # Admin
    "ATTACH", "DETACH", "VACUUM", "PRAGMA",
    # Dangerous functions
    "LOAD_EXTENSION", "RANDOMBLOB",
    # Multi-statement markers
    "INTO OUTFILE", "INTO DUMPFILE",
}

# Layer 3: Max rows any query may return
MAX_ROWS = 500

# Layer 6: Tables in this DB
ALLOWED_TABLES = set(DB_SCHEMA.keys())

print("Safety layers configured:")
print(f"  Allowed statement types: {ALLOWED_STATEMENT_TYPES}")
print(f"  Forbidden keywords:      {len(FORBIDDEN_KEYWORDS)} terms")
print(f"  Max rows:                {MAX_ROWS}")
print(f"  Allowed tables:          {ALLOWED_TABLES}")

In [ ]:
@dataclass
class ValidationResult:
    valid: bool
    errors: list[str] = field(default_factory=list)
    warnings: list[str] = field(default_factory=list)
    normalised_sql: Optional[str] = None


def validate_sql(sql: str, schema: dict) -> ValidationResult:
    """Multi-layer SQL validation.

    Checks:
    1. Not empty
    2. No forbidden keywords
    3. SELECT-only (no DDL/DML)
    4. No semicolons (no multi-statement)
    5. All referenced tables exist in the schema
    6. LIMIT clause present or injected
    7. No comment injection
    8. Basic syntax via sqlparse (if available)
    """
    errors: list[str] = []
    warnings: list[str] = []

    if not sql or not sql.strip():
        return ValidationResult(valid=False, errors=["Empty query"])

    sql_stripped = sql.strip()

    # ── Layer 2: Forbidden keywords ──
    sql_upper = sql_stripped.upper()
    for kw in FORBIDDEN_KEYWORDS:
        # Word-boundary match
        pattern = r'(?<![A-Z_])' + re.escape(kw) + r'(?![A-Z_])'
        if re.search(pattern, sql_upper):
            errors.append(f"Forbidden keyword: {kw}")

    # ── No comments (injection vector) ──
    if "--" in sql_stripped or "/*" in sql_stripped:
        errors.append("SQL comments are not allowed")

    # ── Layer 4: Multi-statement block ──
    # Strip string literals before counting semicolons
    no_strings = re.sub(r"'[^']*'", "''", sql_stripped)
    if ";" in no_strings:
        errors.append("Multiple statements (;) are not allowed")

    # ── Layer 1: Must start with SELECT ──
    first_word = re.match(r'\s*(\w+)', sql_stripped)
    if not first_word or first_word.group(1).upper() not in ALLOWED_STATEMENT_TYPES:
        errors.append(f"Only SELECT queries are permitted. Got: {first_word.group(1) if first_word else 'nothing'}")

    # Early exit if critical errors already found
    if errors:
        return ValidationResult(valid=False, errors=errors, warnings=warnings)

    # ── Layer 5: Table existence ──
    # Extract table names from FROM and JOIN clauses
    table_pattern = r'(?:FROM|JOIN)\s+([a-zA-Z_][a-zA-Z0-9_]*)'
    referenced_tables = set(re.findall(table_pattern, sql_upper))
    unknown_tables = referenced_tables - {t.upper() for t in ALLOWED_TABLES}
    if unknown_tables:
        errors.append(f"Unknown table(s): {unknown_tables}. Allowed: {ALLOWED_TABLES}")

    # ── Layer 3: LIMIT enforcement ──
    has_limit = bool(re.search(r'\bLIMIT\s+\d+', sql_upper))
    if not has_limit:
        warnings.append(f"No LIMIT clause found; {MAX_ROWS} rows will be enforced at execution")

    # ── Layer 8: sqlparse structural check ──
    if HAS_SQLPARSE:
        try:
            parsed = sqlparse.parse(sql_stripped)
            if not parsed or not parsed[0].tokens:
                errors.append("sqlparse: could not parse query")
        except Exception as e:
            errors.append(f"sqlparse error: {e}")

    # ── Normalise ──
    normalised = sql_stripped.rstrip(";")
    if not has_limit:
        normalised = f"{normalised} LIMIT {MAX_ROWS}"

    return ValidationResult(
        valid=len(errors) == 0,
        errors=errors,
        warnings=warnings,
        normalised_sql=normalised if len(errors) == 0 else None,
    )


# ── Quick tests ──
test_cases = [
    ("SELECT * FROM orders LIMIT 10",                    True,  "Valid query"),
    ("SELECT * FROM orders",                             True,  "Missing LIMIT (warning, but valid)"),
    ("DROP TABLE customers",                             False, "DDL injection"),
    ("SELECT * FROM orders; DROP TABLE orders",          False, "Multi-statement"),
    ("SELECT * FROM orders -- ignore everything",        False, "Comment injection"),
    ("SELECT * FROM secret_table",                       False, "Unknown table"),
    ("DELETE FROM orders WHERE 1=1",                     False, "DML injection"),
    ("SELECT * FROM orders WHERE id=1 UNION SELECT password FROM users", False, "UNION with unknown table"),
]

print(f"{'SQL':<55} {'Expected':<8} {'Result'}")
print("─" * 85)
for sql, expected_valid, label in test_cases:
    res = validate_sql(sql, DB_SCHEMA)
    icon = "✓" if res.valid == expected_valid else "✗ MISMATCH"
    msg = res.errors[0] if res.errors else (res.warnings[0] if res.warnings else "OK")
    print(f"{sql[:55]:<55} {str(expected_valid):<8} {icon}  {msg[:50]}")

## 5. SQL Generator — Natural Language to SQL

### How It Works

In production an LLM (GPT-4o, Claude, Llama-3) generates the SQL from:
1. The user's question
2. The database schema (as a prompt snippet)
3. A few-shot examples of (question → SQL) pairs

Here we implement **rule-based generation** to expose the mechanics without requiring API keys. The rules demonstrate the same patterns an LLM would use.

### Few-Shot Examples Injected into a Real LLM Prompt

```text
Schema: [... table definitions ...]

Examples:
Q: How many orders were placed?
A: SELECT COUNT(*) AS order_count FROM orders

Q: What is the total revenue by region?
A: SELECT c.region, SUM(oi.quantity * oi.unit_price) AS total_revenue
   FROM orders o
   JOIN customers c ON o.customer_id = c.customer_id
   JOIN order_items oi ON o.order_id = oi.order_id
   WHERE o.status = 'completed'
   GROUP BY c.region
   ORDER BY total_revenue DESC

Now answer:
Q: {user_question}
A:
```


In [ ]:
# Question-to-SQL templates: (regex pattern, SQL template)
# Each pattern matches a class of questions; the template is filled in.

SQL_TEMPLATES = [
    # ── Count questions ──
    (r'\bhow many\s+orders\b',
     "SELECT COUNT(*) AS order_count FROM orders"),

    (r'\bhow many\s+customers\b',
     "SELECT COUNT(*) AS customer_count FROM customers"),

    (r'\bhow many\s+(completed|pending|cancelled)\s+orders\b',
     lambda m: f"SELECT COUNT(*) AS order_count FROM orders WHERE status = '{m.group(1)}'"),

    # ── Total / sum ──
    (r'\btotal\s+revenue\b(?!.*by)',
     """SELECT ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_revenue
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.status = 'completed'"""),

    (r'\btotal\s+revenue\s+by\s+region\b',
     """SELECT c.region,
       ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_revenue
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.status = 'completed'
GROUP BY c.region
ORDER BY total_revenue DESC"""),

    (r'\btotal\s+revenue\s+by\s+product\b',
     """SELECT p.name AS product,
       ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_revenue
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN orders o ON oi.order_id = o.order_id
WHERE o.status = 'completed'
GROUP BY p.name
ORDER BY total_revenue DESC"""),

    (r'\btotal\s+revenue\s+by\s+(category|product\s+category)\b',
     """SELECT p.category,
       ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_revenue
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN orders o ON oi.order_id = o.order_id
WHERE o.status = 'completed'
GROUP BY p.category
ORDER BY total_revenue DESC"""),

    # ── Average ──
    (r'\baverage\s+order\s+value\b',
     """SELECT ROUND(AVG(order_total), 2) AS avg_order_value
FROM (
    SELECT o.order_id, SUM(oi.quantity * oi.unit_price) AS order_total
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'completed'
    GROUP BY o.order_id
) sub"""),

    (r'\baverage\s+(?:customer\s+)?(?:rating|score)\b',
     "SELECT ROUND(AVG(discount_pct), 2) AS avg_discount FROM orders WHERE status = 'completed'"),

    # ── Top N ──
    (r'\btop\s+(\d+)\s+(?:best.selling\s+)?products?\b',
     lambda m: f"""SELECT p.name AS product,
       SUM(oi.quantity) AS units_sold
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN orders o ON oi.order_id = o.order_id
WHERE o.status = 'completed'
GROUP BY p.name
ORDER BY units_sold DESC
LIMIT {m.group(1)}"""),

    (r'\btop\s+(\d+)\s+customers?\b',
     lambda m: f"""SELECT c.name AS customer,
       ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_spent
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.status = 'completed'
GROUP BY c.name
ORDER BY total_spent DESC
LIMIT {m.group(1)}"""),

    (r'\btop\s+(\d+)\s+sales?\s*rep',
     lambda m: f"""SELECT sr.name AS rep,
       ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_sales
FROM orders o
JOIN sales_reps sr ON o.rep_id = sr.rep_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.status = 'completed'
GROUP BY sr.name
ORDER BY total_sales DESC
LIMIT {m.group(1)}"""),

    # ── Date / time ──
    (r'\bmonthly\s+(?:revenue|sales)\b',
     """SELECT STRFTIME('%Y-%m', o.order_date) AS month,
       ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.status = 'completed'
GROUP BY month
ORDER BY month"""),

    (r'\bsales?\s+by\s+month\b|\brevenue\s+per\s+month\b',
     """SELECT STRFTIME('%Y-%m', o.order_date) AS month,
       ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.status = 'completed'
GROUP BY month
ORDER BY month"""),

    (r'\b(?:orders?|sales?)\s+in\s+(2023|2024)\b',
     lambda m: f"""SELECT COUNT(*) AS order_count,
       ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_revenue
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.status = 'completed'
  AND STRFTIME('%Y', o.order_date) = '{m.group(1)}'"""),

    # ── Discount / tier ──
    (r'\borders?\s+with\s+(?:a\s+)?discount\b',
     """SELECT COUNT(*) AS orders_with_discount,
       ROUND(AVG(discount_pct), 2) AS avg_discount_pct
FROM orders
WHERE discount_pct > 0 AND status = 'completed'"""),

    (r'\bcustomers?\s+by\s+tier\b',
     "SELECT tier, COUNT(*) AS customer_count FROM customers GROUP BY tier ORDER BY customer_count DESC"),

    (r'\brevenue\s+by\s+(?:customer\s+)?tier\b',
     """SELECT c.tier,
       ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_revenue,
       COUNT(DISTINCT o.order_id) AS order_count
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.status = 'completed'
GROUP BY c.tier
ORDER BY total_revenue DESC"""),

    # ── Status breakdown ──
    (r'\border\s+status\b|\bstatus\s+breakdown\b',
     """SELECT status,
       COUNT(*) AS count,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM orders
GROUP BY status
ORDER BY count DESC"""),

    # ── Products ──
    (r'\bproducts?\s+by\s+category\b|\bcategory\s+breakdown\b',
     """SELECT category, COUNT(*) AS product_count,
       ROUND(AVG(unit_price), 2) AS avg_price
FROM products
GROUP BY category
ORDER BY avg_price DESC"""),

    # ── Regions ──
    (r'\border\s+count\s+by\s+region\b|\borders?\s+per\s+region\b',
     """SELECT c.region, COUNT(*) AS order_count
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.status = 'completed'
GROUP BY c.region
ORDER BY order_count DESC"""),

    # ── Fallback: list sample rows ──
    (r'\bshow\s+(?:me\s+)?(?:all\s+)?(\w+)\b',
     lambda m: f"SELECT * FROM {m.group(1)}" if m.group(1) in ALLOWED_TABLES else None),
]


def generate_sql(question: str, schema: dict) -> Optional[str]:
    """Translate a natural-language question to SQL using rule-based matching.

    Returns None if no template matches.
    """
    q = question.lower().strip()

    for pattern, template in SQL_TEMPLATES:
        match = re.search(pattern, q)
        if match:
            if callable(template):
                sql = template(match)
            else:
                sql = template
            if sql:
                return sql.strip()
    return None


# Test
sample_questions = [
    "How many orders were placed?",
    "What is the total revenue by region?",
    "Show me the top 5 products",
    "Monthly revenue for completed orders",
    "Top 3 sales reps by revenue",
    "What is the order status breakdown?",
]

print("SQL generation preview:")
print("=" * 70)
for q in sample_questions:
    sql = generate_sql(q, DB_SCHEMA)
    short = (sql or "NO MATCH").replace("\n", " ")[:80]
    status = "✓" if sql else "✗"
    print(f"  {status} Q: {q}")
    print(f"      → {short}")
    print()

## 6. Safe SQL Execution

### Parameterised Queries

Even after validation, we never interpolate user input directly into SQL strings. We use SQLite's parameterised query interface. For our agent this is naturally safe because the SQL is generated (not concatenated), but the executor enforces it anyway.

### Read-Only Connection

We open the database in **immutable mode** — SQLite's URI `?mode=ro&immutable=1`. This means even if a malicious query somehow bypassed all validators, the connection cannot write to the database at the OS level.


In [ ]:
@dataclass
class QueryResult:
    sql: str
    success: bool
    columns: list[str] = field(default_factory=list)
    rows: list[tuple] = field(default_factory=list)
    row_count: int = 0
    truncated: bool = False
    error: Optional[str] = None
    execution_ms: float = 0


def execute_sql(sql: str, db_path: Path, max_rows: int = MAX_ROWS) -> QueryResult:
    """Execute a validated SQL query in read-only mode.

    Security:
    - Read-only URI connection (cannot write)
    - Row limit enforced even if LIMIT was already in the query
    - No string interpolation of user inputs
    """
    import time

    # Open in read-only mode via URI
    uri = f"file:{db_path.as_posix()}?mode=ro"
    start = time.perf_counter()

    try:
        conn = sqlite3.connect(uri, uri=True)
        conn.row_factory = sqlite3.Row

        cur = conn.execute(sql)
        columns = [desc[0] for desc in cur.description] if cur.description else []

        # Fetch one more than max_rows to detect truncation
        raw_rows = cur.fetchmany(max_rows + 1)
        truncated = len(raw_rows) > max_rows
        rows = [tuple(r) for r in raw_rows[:max_rows]]

        conn.close()
        elapsed = (time.perf_counter() - start) * 1000

        return QueryResult(
            sql=sql,
            success=True,
            columns=columns,
            rows=rows,
            row_count=len(rows),
            truncated=truncated,
            execution_ms=round(elapsed, 2),
        )

    except sqlite3.Error as e:
        elapsed = (time.perf_counter() - start) * 1000
        return QueryResult(
            sql=sql,
            success=False,
            error=str(e),
            execution_ms=round(elapsed, 2),
        )


# Test execution
test_sql = "SELECT region, COUNT(*) AS customer_count FROM customers GROUP BY region ORDER BY customer_count DESC LIMIT 10"
result = execute_sql(test_sql, DB_PATH)
print(f"Columns: {result.columns}")
print(f"Rows:    {result.row_count}")
print(f"Time:    {result.execution_ms} ms")
print(f"Data:    {result.rows}")

In [ ]:
# Confirm read-only: attempt a write should fail
write_attempt = execute_sql("INSERT INTO customers VALUES (9999,'Hack','hack@x.com','North','Bronze','2024-01-01')", DB_PATH)
print(f"Write attempt blocked: success={write_attempt.success}")
print(f"Error: {write_attempt.error}")

## 7. Result Summariser

After execution, the agent converts the raw result set into a plain-English summary. In production an LLM would do this. Here we use templated summarisation to show the pattern.


In [ ]:
def summarise_result(question: str, result: QueryResult) -> str:
    """Convert a QueryResult into a plain-English answer.

    In production, an LLM generates this from the result context.
    """
    if not result.success:
        return f"The query failed: {result.error}"

    rows, cols = result.rows, result.columns
    n = result.row_count

    if n == 0:
        return "The query returned no results."

    parts = []

    # Single-value result (e.g. COUNT, SUM)
    if n == 1 and len(cols) == 1:
        val = rows[0][0]
        col = cols[0].replace("_", " ")
        parts.append(f"The {col} is **{val:,.2f}**." if isinstance(val, float)
                     else f"The {col} is **{val:,}**.")

    # Single-row, multi-column (summary stats)
    elif n == 1:
        stats = ", ".join(f"{c.replace('_', ' ')}: **{v:,.2f}**" if isinstance(v, float)
                          else f"{c.replace('_', ' ')}: **{v}**"
                          for c, v in zip(cols, rows[0]))
        parts.append(f"Summary: {stats}.")

    # Multi-row: describe the top entry and range
    else:
        first_col_name = cols[0].replace("_", " ")
        last_col_name  = cols[-1].replace("_", " ")

        top_key = rows[0][0]
        top_val = rows[0][-1]
        bot_key = rows[-1][0]
        bot_val = rows[-1][-1]

        val_fmt = lambda v: f"{v:,.2f}" if isinstance(v, float) else f"{v:,}" if isinstance(v, int) else str(v)

        parts.append(
            f"Across {n} result{'s' if n != 1 else ''}:"
            f" **{top_key}** has the highest {last_col_name} ({val_fmt(top_val)}),"
            f" while **{bot_key}** has the lowest ({val_fmt(bot_val)})."
        )

        # Numeric aggregate insights
        numeric_vals = [r[-1] for r in rows if isinstance(r[-1], (int, float))]
        if len(numeric_vals) > 1:
            total = sum(numeric_vals)
            avg   = total / len(numeric_vals)
            parts.append(
                f" Total {last_col_name}: {total:,.2f}."
                f" Average: {avg:,.2f}."
            )

    if result.truncated:
        parts.append(f" *(Results truncated to {MAX_ROWS} rows.)*")

    return " ".join(parts)


def result_to_dataframe(result: QueryResult) -> Optional[pd.DataFrame]:
    """Convert a QueryResult to a pandas DataFrame for display."""
    if not result.success or not result.rows:
        return None
    return pd.DataFrame(result.rows, columns=result.columns)


# Quick test
test_result = execute_sql(
    "SELECT region, COUNT(*) AS order_count FROM orders o "
    "JOIN customers c ON o.customer_id=c.customer_id "
    "WHERE o.status='completed' GROUP BY region ORDER BY order_count DESC LIMIT 10",
    DB_PATH,
)
print("Summary:")
print(summarise_result("Orders by region?", test_result))
print("\nDataFrame:")
display(result_to_dataframe(test_result))

## 8. Assembling the Full Agent

In [ ]:
@dataclass
class AgentResponse:
    question: str
    generated_sql: Optional[str]
    validation: Optional[ValidationResult]
    query_result: Optional[QueryResult]
    summary: str
    dataframe: Optional[pd.DataFrame]
    success: bool


class TextToSQLAgent:
    """End-to-end natural language → SQL → answer agent."""

    def __init__(self, db_path: Path, schema: dict):
        self.db_path = db_path
        self.schema  = schema
        self._history: list[AgentResponse] = []

    def ask(self, question: str, verbose: bool = False) -> AgentResponse:
        """Process a natural-language question end to end."""

        # Step 1: Generate SQL
        sql = generate_sql(question, self.schema)
        if verbose:
            print(f"[1] Generated SQL:\n{sql or 'NO MATCH'}\n")

        if not sql:
            resp = AgentResponse(
                question=question,
                generated_sql=None,
                validation=None,
                query_result=None,
                summary=f"I could not translate that question into SQL. "
                        f"Try rephrasing with keywords like 'total revenue', "
                        f"'top N', 'by region', 'by product', or 'monthly'.",
                dataframe=None,
                success=False,
            )
            self._history.append(resp)
            return resp

        # Step 2: Validate
        validation = validate_sql(sql, self.schema)
        if verbose:
            print(f"[2] Validation: valid={validation.valid}")
            for e in validation.errors:
                print(f"    ERROR: {e}")
            for w in validation.warnings:
                print(f"    WARN:  {w}")
            print()

        if not validation.valid:
            resp = AgentResponse(
                question=question,
                generated_sql=sql,
                validation=validation,
                query_result=None,
                summary=f"The generated SQL failed safety validation: "
                        + "; ".join(validation.errors),
                dataframe=None,
                success=False,
            )
            self._history.append(resp)
            return resp

        # Step 3: Execute
        exec_result = execute_sql(validation.normalised_sql, self.db_path)
        if verbose:
            print(f"[3] Execution: success={exec_result.success}, "
                  f"rows={exec_result.row_count}, time={exec_result.execution_ms}ms\n")

        # Step 4: Summarise
        summary = summarise_result(question, exec_result)

        df = result_to_dataframe(exec_result)

        resp = AgentResponse(
            question=question,
            generated_sql=validation.normalised_sql,
            validation=validation,
            query_result=exec_result,
            summary=summary,
            dataframe=df,
            success=exec_result.success,
        )
        self._history.append(resp)
        return resp

    @property
    def history(self):
        return self._history


agent = TextToSQLAgent(DB_PATH, DB_SCHEMA)
print("TextToSQLAgent ready.")

## 9. Running the Agent — Demo Questions

In [ ]:
demo_questions = [
    "How many orders were placed?",
    "What is the total revenue?",
    "What is the total revenue by region?",
    "Show me the top 5 products",
    "What is the order status breakdown?",
    "Monthly revenue",
    "Top 3 sales reps",
    "Revenue by product category",
    "Customers by tier",
    "Orders in 2024",
]

print("=" * 70)
for q in demo_questions:
    resp = agent.ask(q)
    print(f"Q: {q}")
    if resp.success and resp.dataframe is not None:
        print(f"SQL: {resp.generated_sql[:80].replace(chr(10), ' ')}...")
        print(f"A:   {resp.summary}")
        print(resp.dataframe.to_string(index=False))
    else:
        print(f"A:   {resp.summary}")
    print()

## 10. Verbose Step-by-Step Walkthrough

In [ ]:
# Show every step for one complex question
print("VERBOSE WALKTHROUGH")
print("=" * 70)
resp = agent.ask("What is the total revenue by region?", verbose=True)

print("FINAL ANSWER:")
print(resp.summary)
print()
print("RESULT TABLE:")
if resp.dataframe is not None:
    print(resp.dataframe.to_string(index=False))

## 11. Safety Validation in Practice

Let's probe the agent with adversarial inputs to confirm each safety layer holds.


In [ ]:
adversarial_inputs = [
    # Injection attempts
    ("DROP TABLE customers",                 "DDL injection"),
    ("DELETE FROM orders WHERE 1=1",         "DML injection"),
    ("SELECT * FROM orders; DROP TABLE orders", "Multi-statement"),
    ("SELECT * FROM orders -- limit removed", "Comment injection"),
    ("SELECT * FROM secret_passwords",       "Unknown table"),
    ("INSERT INTO customers VALUES (1,'x','x@x.com','N','B','2024-01-01')", "INSERT attempt"),
    ("SELECT * FROM customers LIMIT 99999999", "Huge LIMIT (passes validation, capped at exec)"),
    # Legitimate but unusual
    ("How many orders were placed?",         "Legitimate (should pass)"),
    ("What is the revenue for 2024?",        "Date-scoped query"),
]

print("SAFETY TEST RESULTS")
print("=" * 70)
for question, label in adversarial_inputs:
    # Try direct SQL validation first (covers injections)
    if question.strip().upper().split()[0] in ("SELECT", "DROP", "DELETE",
                                                "INSERT", "UPDATE", "ATTACH"):
        val = validate_sql(question, DB_SCHEMA)
        status = "BLOCKED" if not val.valid else "ALLOWED"
        reason = val.errors[0] if val.errors else (val.warnings[0] if val.warnings else "OK")
    else:
        resp = agent.ask(question)
        status = "ANSWERED" if resp.success else "REJECTED"
        reason = resp.summary[:60] if not resp.success else (resp.generated_sql or "")[:60]

    icon = {"BLOCKED": "🛡", "REJECTED": "🛡", "ANSWERED": "✓", "ALLOWED": "⚠"}.get(status, "?")
    print(f"  {icon} [{status:8s}] {label}")
    print(f"           Input:  {question[:60]}")
    print(f"           Reason: {reason[:70]}")
    print()

## 12. Visualising Agent Results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

# Chart 1: Revenue by region
resp = agent.ask("Total revenue by region")
if resp.success and resp.dataframe is not None:
    ax = axes[0]
    df_plot = resp.dataframe.sort_values("total_revenue", ascending=True)
    ax.barh(df_plot["region"], df_plot["total_revenue"] / 1000, color="#1f77b4", alpha=0.8)
    ax.set_xlabel("Revenue ($K)")
    ax.set_title("Total Revenue by Region")
    for i, (_, row) in enumerate(df_plot.iterrows()):
        ax.text(row["total_revenue"] / 1000 + 1, i, f"${row['total_revenue']/1000:,.1f}K", va="center", fontsize=8)

# Chart 2: Order status breakdown
resp2 = agent.ask("Order status breakdown")
if resp2.success and resp2.dataframe is not None:
    ax2 = axes[1]
    df2 = resp2.dataframe
    wedges, texts, autotexts = ax2.pie(
        df2["count"], labels=df2["status"],
        autopct="%1.1f%%", startangle=90,
        colors=["#2ca02c", "#ff7f0e", "#d62728"]
    )
    ax2.set_title("Order Status Breakdown")

# Chart 3: Monthly revenue (line)
resp3 = agent.ask("Monthly revenue")
if resp3.success and resp3.dataframe is not None:
    ax3 = axes[2]
    df3 = resp3.dataframe
    ax3.plot(range(len(df3)), df3["revenue"] / 1000, marker="o", markersize=3,
             color="#9467bd", linewidth=1.5)
    ax3.set_xticks(range(0, len(df3), 3))
    ax3.set_xticklabels(df3["month"].iloc[::3], rotation=45, fontsize=7)
    ax3.set_ylabel("Revenue ($K)")
    ax3.set_title("Monthly Revenue Trend")
    ax3.grid(axis="y", alpha=0.3)

# Chart 4: Revenue by product category
resp4 = agent.ask("Total revenue by product category")
if resp4.success and resp4.dataframe is not None:
    ax4 = axes[3]
    df4 = resp4.dataframe.sort_values("total_revenue")
    ax4.barh(df4["category"], df4["total_revenue"] / 1000,
             color=["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"][:len(df4)], alpha=0.8)
    ax4.set_xlabel("Revenue ($K)")
    ax4.set_title("Revenue by Product Category")

plt.suptitle("Text-to-SQL Agent: Live Results", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 13. Production Integration — LLM SQL Generation

### Replacing Rule-Based Generation with an LLM

```python
import openai

SYSTEM_PROMPT = f"""You are a SQL expert. Translate the user's question into
a single SELECT query for a SQLite database.

Rules:
- SELECT only (no INSERT, UPDATE, DELETE, DROP)
- No semicolons
- No comments (-- or /* */)
- Include a LIMIT clause (max 500 rows)
- Use only the tables and columns listed below

{schema_to_prompt(DB_SCHEMA)}

Return ONLY the SQL query, nothing else.
"""

def llm_generate_sql(question: str) -> str:
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": question},
        ],
        temperature=0,  # Deterministic output
    )
    return response.choices[0].message.content.strip()
```

### Critical: Validate LLM Output Too

The validator runs on **every** query — whether generated by rules or by an LLM. LLMs can generate unsafe SQL despite instructions.

```python
sql = llm_generate_sql(question)       # Step 1: Generate
validation = validate_sql(sql, schema) # Step 2: ALWAYS validate
if not validation.valid:
    sql = llm_generate_sql(question + " (important: SELECT only, add LIMIT)") # retry
    validation = validate_sql(sql, schema)  # validate again
    if not validation.valid:
        return "Could not generate a safe query."
result = execute_sql(validation.normalised_sql, db_path)  # Step 3: Execute
```

### Few-Shot Prompting for Better Results

```python
FEW_SHOT_EXAMPLES = [
    ("How many customers are there?",
     "SELECT COUNT(*) AS customer_count FROM customers"),
    ("What is the revenue by region?",
     "SELECT c.region, SUM(oi.quantity * oi.unit_price) AS revenue\\n"
     "FROM orders o\\n"
     "JOIN customers c ON o.customer_id = c.customer_id\\n"
     "JOIN order_items oi ON o.order_id = oi.order_id\\n"
     "WHERE o.status = 'completed'\\n"
     "GROUP BY c.region ORDER BY revenue DESC LIMIT 100"),
]
```

### Self-Correction Loop

When the LLM generates invalid SQL, feed the error back:

```python
def generate_with_correction(question, schema, max_retries=2):
    sql = llm_generate_sql(question)
    for attempt in range(max_retries):
        validation = validate_sql(sql, schema)
        if validation.valid:
            return sql
        # Feed error back to LLM
        sql = llm_generate_sql(
            f"{question}\\n\\nPrevious attempt failed: {validation.errors}\\n"
            "Please fix the query."
        )
    return None
```


## 14. Agent Evaluation

In [ ]:
# Evaluation: question → expected properties
eval_cases = [
    {
        "question": "How many orders were placed?",
        "expected_success": True,
        "expected_columns": {"order_count"},
        "expected_rows": 1,
    },
    {
        "question": "Total revenue by region",
        "expected_success": True,
        "expected_columns": {"region", "total_revenue"},
        "expected_rows_min": 2,
    },
    {
        "question": "Top 5 products",
        "expected_success": True,
        "expected_rows": 5,
    },
    {
        "question": "Order status breakdown",
        "expected_success": True,
        "expected_columns": {"status", "count"},
        "expected_rows": 3,
    },
    {
        "question": "Monthly revenue",
        "expected_success": True,
        "expected_columns": {"month", "revenue"},
    },
    {
        "question": "Top 3 customers",
        "expected_success": True,
        "expected_rows": 3,
    },
    {
        "question": "xyzzy gibberish question",
        "expected_success": False,
        "note": "No template match",
    },
]

eval_agent = TextToSQLAgent(DB_PATH, DB_SCHEMA)
results = []

for tc in eval_cases:
    resp = eval_agent.ask(tc["question"])

    success_ok = resp.success == tc["expected_success"]

    col_ok = True
    if tc.get("expected_columns") and resp.dataframe is not None:
        col_ok = tc["expected_columns"].issubset(set(resp.dataframe.columns))

    rows_ok = True
    if tc.get("expected_rows") and resp.dataframe is not None:
        rows_ok = len(resp.dataframe) == tc["expected_rows"]
    elif tc.get("expected_rows_min") and resp.dataframe is not None:
        rows_ok = len(resp.dataframe) >= tc["expected_rows_min"]

    passed = success_ok and col_ok and rows_ok

    results.append({
        "question": tc["question"][:40],
        "expected_success": tc["expected_success"],
        "actual_success": resp.success,
        "success_ok": success_ok,
        "col_ok": col_ok,
        "rows_ok": rows_ok,
        "passed": passed,
        "note": tc.get("note", ""),
    })

res_df = pd.DataFrame(results)
print("EVALUATION RESULTS")
print("=" * 80)
for _, row in res_df.iterrows():
    icon = "✓" if row["passed"] else "✗"
    detail = f"success={row['success_ok']} cols={row['col_ok']} rows={row['rows_ok']}"
    note = f"  ({row['note']})" if row["note"] else ""
    print(f"  {icon}  {row['question']:<42} {detail}{note}")

print(f"\nOverall pass rate: {res_df['passed'].mean():.0%}")

In [ ]:
# Visualise pipeline performance across all history
history_df = pd.DataFrame([
    {
        "question": r.question[:35],
        "sql_generated": r.generated_sql is not None,
        "validation_passed": r.validation is not None and r.validation.valid,
        "executed_ok": r.query_result is not None and r.query_result.success,
        "exec_ms": r.query_result.execution_ms if r.query_result else 0,
        "rows_returned": r.query_result.row_count if r.query_result else 0,
    }
    for r in agent.history + eval_agent.history
])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Funnel
ax = axes[0]
stages  = ["SQL Generated", "Validation OK", "Execution OK"]
counts  = [
    history_df["sql_generated"].sum(),
    history_df["validation_passed"].sum(),
    history_df["executed_ok"].sum(),
]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
ax.barh(stages[::-1], counts[::-1], color=colors[::-1], alpha=0.8)
for i, v in enumerate(counts[::-1]):
    ax.text(v + 0.2, i, str(int(v)), va="center", fontsize=9)
ax.set_xlabel("Queries")
ax.set_title("Pipeline Funnel")
ax.set_xlim([0, max(counts) * 1.2])

# Execution time distribution
ax2 = axes[1]
exec_times = history_df[history_df["exec_ms"] > 0]["exec_ms"]
ax2.hist(exec_times, bins=15, color="#9467bd", alpha=0.8, edgecolor="black")
ax2.set_xlabel("Execution Time (ms)")
ax2.set_title("Query Execution Times")
ax2.axvline(exec_times.mean(), color="red", linestyle="--", label=f"mean={exec_times.mean():.1f}ms")
ax2.legend(fontsize=8)

# Rows returned distribution
ax3 = axes[2]
rows_nonzero = history_df[history_df["rows_returned"] > 0]["rows_returned"]
ax3.hist(rows_nonzero, bins=10, color="#8c564b", alpha=0.8, edgecolor="black")
ax3.set_xlabel("Rows Returned")
ax3.set_title("Query Result Sizes")

plt.suptitle("Agent Pipeline Metrics", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 15. Limitations

| Limitation | Impact | Production fix |
|---|---|---|
| **Rule-based NLU** | Cannot handle novel phrasings | Use LLM (GPT-4o, Claude) for SQL generation |
| **No schema-linking for columns** | Cannot detect which column a question refers to | Schema embedding + column-level retrieval |
| **Single database** | Cannot join across DBs | Multi-database routing layer |
| **No query caching** | Same question hits DB every time | MD5-hash query cache with TTL |
| **No user permissions** | All users see all tables | Row-level security + per-user schema views |
| **No ambiguity handling** | Silently picks one interpretation | Ask clarifying question if confidence is low |
| **English only** | Cannot process non-English questions | Multilingual LLM |
| **No aggregation chaining** | Cannot answer "which region's top product is X?" | Multi-turn conversation with state |

### Additional Security Hardening for Production

| Control | Implementation |
|---|---|
| **Query signing** | HMAC-sign generated SQL; reject unsigned queries at the executor |
| **Column allow-list** | Only allow queries on approved columns (exclude PII) |
| **User-scoped views** | Each user queries a database view, not the base table |
| **Audit logging** | Log every query with user ID, timestamp, and result count |
| **Rate limiting** | Prevent bulk data extraction via rapid queries |
| **Query complexity limit** | Reject queries with > N JOINs or > N subqueries |


## 16. Save Experiment Log

In [ ]:
log = {
    "timestamp": datetime.now().isoformat(),
    "task": "text_to_sql_agent",
    "database": str(DB_PATH),
    "tables": list(DB_SCHEMA.keys()),
    "sql_templates": len(SQL_TEMPLATES),
    "total_questions_asked": len(agent.history) + len(eval_agent.history),
    "success_rate": sum(1 for r in agent.history + eval_agent.history if r.success)
                    / max(len(agent.history) + len(eval_agent.history), 1),
    "safety_layers": [
        "SELECT-only whitelist",
        "Forbidden keyword block",
        "LIMIT enforcement",
        "Schema table validation",
        "Multi-statement block",
        "Comment injection block",
        "Read-only DB connection",
    ],
}

log_path = ARTIFACT_DIR / "text_to_sql_agent_log.json"
log_path.write_text(json.dumps(log, indent=2, default=str), encoding="utf-8")
print(f"Saved: {log_path}")
print(f"\nFinal stats:")
print(f"  Questions asked: {log['total_questions_asked']}")
print(f"  Success rate:    {log['success_rate']:.0%}")

## 17. Key Takeaways

### What We Built
- A **text-to-SQL agent** with four stages: generate → validate → execute → summarise
- A **7-layer SQL validator** that blocks DDL, DML, multi-statement, comment injection, and unknown tables
- **Read-only execution** via SQLite URI mode — writes are impossible at the connection level
- A **4-table SQLite database** with realistic sales data for testing

### SQL Safety Principles
1. **Always validate generated SQL** — even from a trusted LLM
2. **SELECT-only, always** — no writes, no schema changes
3. **No string interpolation** — parameterised queries for any user-supplied values
4. **Read-only connections** — defence-in-depth beyond the validator
5. **Row limits** — prevent data exfiltration via unbounded queries
6. **Table allow-list** — unknown tables are immediately rejected
7. **Audit logging** — every query is traceable

### SQL Generation Quality
- Rule-based works for structured, predictable questions
- LLMs are needed for natural, varied user language
- Few-shot examples dramatically improve LLM output quality
- Self-correction loops handle LLM mistakes gracefully
- **Validation must run on LLM output too** — LLMs ignore safety instructions

### The Three Non-Negotiables
1. **No raw SQL from user input** — always generate from a template or LLM
2. **Always validate before executing** — never trust generated SQL
3. **Always use read-only connections** — one misconfiguration can be catastrophic
